In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html#sklearn.ensemble.HistGradientBoostingRegressor

# df = pd.read_csv('/Users/nrcase/CSC522/CSC522-Project/dataset_with_labels.csv')
df = pd.read_csv('/Users/hannah/git/CSC522-Project/dataset_with_labels.csv')
df.head()

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",1,1,0,NaN,2025-02-17,98,False,...,-7.777,0,0.0304,0.3080,0.0000,0.122,0.535,157.969,3,Lower
1,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",2,1,4,NaN,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.0000,0.248,0.576,138.008,4,About_Average
2,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,3,-2,8,NaN,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.0000,0.141,0.214,101.061,4,Higher
3,4wJ5Qq0jBN4ajy7ouZIV1c,APT.,"ROSÉ, Bruno Mars",4,0,-2,NaN,2025-02-17,89,False,...,-4.477,0,0.2600,0.0283,0.0000,0.355,0.939,149.027,4,Higher
4,6dOtVTDdiauQNBQEDOtlAB,BIRDS OF A FEATHER,Billie Eilish,5,1,-2,NaN,2025-02-17,96,False,...,-10.171,1,0.0358,0.2000,0.0608,0.117,0.438,104.978,4,About_Average


In [3]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'popularity',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement', 'popularity',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    HistGradientBoostingRegressor()
)

# Tuning Hyperparameters

In [5]:
def explore_single_hp_values(pipeline, param_name, param_values, X_train, y_train, k_fold, scoring):
    param_grid = {param_name: param_values}
    grid_search = GridSearchCV(pipeline, param_grid, cv=k_fold, scoring=scoring)
    grid_search.fit(X_train, y_train)
    result_columns = [f"param_{param_name}", "mean_test_score", "std_test_score", "rank_test_score"]
    return pd.DataFrame(grid_search.cv_results_)[result_columns]
  
import seaborn as sns
import matplotlib.pyplot as plt

def plot_gridsearch_heatmap(results_df, x_param, y_param, score='neg_mean_squared_error'):    
    # Pivot the table to format it for a heatmap
    heatmap_data = results_df.pivot(index=f'param_{y_param}', columns=f'param_{x_param}', values=score)
    
    # Plot heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(heatmap_data, annot=True, cmap='viridis', fmt='.3f', linewidths=0.5)
    plt.title(f'Grid Search Results: {score}')
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.show()

In [22]:
# Hyperparameter Tuning
hgbr_step_name = pipeline.steps[1][0]
step_prefix = '__'
hgbr_step_prefix = f"{hgbr_step_name}{step_prefix}"

# Parameter Keys
loss_key = hgbr_step_prefix + "loss"
max_leaf_key = hgbr_step_prefix + "max_leaf_nodes"
max_depth_key = hgbr_step_prefix + "max_depth"

# hgbr_param_keys = HistGradientBoostingRegressor().get_params().keys()
# for key in [loss_key, learning_rate_key, max_leaf_key, max_depth_key]:
#     assert key.startswith(hgbr_step_name), f"Key {key} should start with {hgbr_step_name}"
#     assert "__" in key, f"Key {key} should be connected by a double underscope __"
#     assert key[key.index("__") + 2:] in hgbr_param_keys, f"Key {key} should end with a DT parameter name"

from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

k_fold = KFold(n_splits=5, shuffle=True, random_state=42)

# Define the param_grid and grid_search
param_grid = {
  loss_key: ['squared_error', 'absolute_error'],
  max_leaf_key: [20, 25, 30, 35, 40],
  max_depth_key:[5,15,25,35]
}
grid_search = GridSearchCV(pipeline, param_grid, scoring="neg_mean_squared_error", cv=k_fold)
grid_search.fit(X_train, y_train)

# Print the best parameters
grid_search.best_params_
grid_search.best_score_

np.float64(-1.8373022946817115e-07)

In [29]:
# results = pd.DataFrame(grid_search.cv_results_)
# results = results[results['param_' + loss_key] == 'squared_error']
# plot_gridsearch_heatmap(results, max_leaf_key, max_depth_key)
grid_search.best_params_

{'histgradientboostingregressor__loss': 'squared_error',
 'histgradientboostingregressor__max_depth': 25,
 'histgradientboostingregressor__max_leaf_nodes': 40}

In [31]:
print("Pre-tuned")
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

Pre-tuned
MSE:  2.399855637275111e-07
RMSE:  0.0004898832143761522


# Fitting Pipeline w/ Tuned HPs

In [6]:
pipeline_tuned = make_pipeline(
    preprocessing,
    HistGradientBoostingRegressor(loss='squared_error', max_depth=25, max_leaf_nodes=40)
)

pipeline_tuned.fit(X_train, y_train)
y_pred = pipeline_tuned.predict(X_test)

print("Tuned!")
print("MSE: ", mean_squared_error(y_test, y_pred))
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

Tuned!
MSE:  1.889207900853808e-07
RMSE:  0.00043465019278194364


In [34]:
pred = pd.DataFrame(y_pred).value_counts()
test = pd.DataFrame(y_test).value_counts()

print(pred.describe())
print(test.describe())

count      111.000000
mean      3114.207207
std       3591.957043
min          1.000000
25%         49.000000
50%       1329.000000
75%       5716.500000
max      12416.000000
Name: count, dtype: float64
count      101.000000
mean      3422.544554
std       3623.056051
min         16.000000
25%         84.000000
50%       1794.000000
75%       5767.000000
max      12416.000000
Name: count, dtype: float64
